<a href="https://colab.research.google.com/github/ksuplee/AI_Agent/blob/main/07_2_Chain_QA_Agent_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 07-2: Chain 기반 Q&A 에이전트 구현
이 노트북에서는 LangChain의 가장 핵심적인 단위인 **Chain**을 구성하고, 이를 통해 단순 질의응답(Q&A) 에이전트를 만드는 실습을 진행합니다.

### 1. 환경 준비
필요한 라이브러리를 설치합니다.

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface transformers accelerate

In [2]:
!pip install -q google-generativeai langchain-google-genai

### 2. Gemini LLM 로드

Google Gemini 모델을 사용하여 더 나은 답변을 시도할 수 있습니다. Gemini 모델을 사용하려면 `google-generativeai` 라이브러리를 설치하고 API 키를 설정해야 합니다.

Gemini API를 사용하려면 API 키가 필요합니다.

아직 키가 없다면 Google AI Studio에서 키를 생성하세요.

1. Google AI Studio 접속
먼저 공식 사이트(https://aistudio.google.com)에 접속합니다. 사용 중인 구글 계정으로 로그인해 주세요.

2. 서비스 약관 동의
처음 접속하신 경우, 생성형 AI 사용을 위한 서비스 약관 동의 팝업이 뜹니다. 내용을 확인하신 후 'Accept' 또는 'Continue' 버튼을 클릭하여 메인 대시보드로 진입합니다.

3. API 키 메뉴 이동  
    - [대시보드] 왼쪽 상단 메뉴 바에서 [Get API key] 항목을 클릭합니다.

4. API 키 생성: 화면 중앙에 보이는 버튼 중 하나를 선택합니다.  

    - [Create API key] in new project: 새로운 프로젝트를 생성하면서 키를 발급받습니다. (처음 만드시는 분들께 권장)

    - Create API key in existing project: 기존에 사용하던 Google Cloud 프로젝트가 있다면 해당 프로젝트를 선택하여 키를 생성합니다.

5. 키 복사 및 안전한 보관:  
팝업창에 생성된 **긴 문자열(API Key)**이 나타납니다. 'Copy' 버튼을 눌러 복사한 뒤, 메모장이나 환경 변수 설정 등 안전한 곳에 저장해 두세요.

    ⚠️ 주의: API 키는 비밀번호와 같습니다. GitHub 같은 공개 저장소에 코드를 올릴 때 키가 노출되지 않도록 주의하세요!

6. 팁: 요금 및 제한 사항 (무료 티어 기준)  

    - Gemini 1.5 Flash: 속도가 빠르고 무료 사용량이 넉넉하여 테스트용으로 좋습니다.  
    - Gemini 1.5 Pro: 복잡한 추론에 적합하지만, 무료 티어에서는 분당 요청 횟수(RPM) 제한이 더 타이트합니다.  
    - 개인정보: 무료 등급 사용 시 입력한 데이터는 모델 학습에 사용될 수 있으므로 민감한 정보는 입력하지 않는 것이 좋습니다.  

Colab에서는 왼쪽 패널의 "🔑" 아래에 키를 `GOOGLE_API_KEY`라는 이름으로 Secrets Manager에 추가하세요. 그런 다음 키를 SDK에 전달합니다.

In [3]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

print("✅ Gemini API 설정 완료")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ Gemini API 설정 완료


이제 Gemini 모델을 초기화하여 `llm` 변수에 할당하겠습니다. 이렇게 하면 기존 Hugging Face 모델 대신 Gemini 모델이 사용됩니다.

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Gemini 모델 초기화
llm = ChatGoogleGenerativeAI(model='gemini-flash-latest', api_key=GOOGLE_API_KEY)

print("✅ Gemini LLM 로드 완료")

✅ Gemini LLM 로드 완료


이제 `qa_chain`은 새로 로드된 Gemini LLM을 사용하게 됩니다. 다시 질문을 실행하여 답변을 확인해보겠습니다.

### 3. PromptTemplate 정의
에이전트의 사고 규칙(역할, 스타일)을 설정합니다.

In [5]:
from langchain_core.prompts import PromptTemplate

# 에이전트 페르소나 및 지시문 설정
template = """너는 친절하고 똑똑한 AI 도우미야.
다음 참고 문서를 활용하여 질문에 대해 핵심 위주로 명확하게 답변해줘.

참고 문서: {context}
질문: {question}

답변:"""

prompt = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)
print("✅ PromptTemplate 구성 완료")

✅ PromptTemplate 구성 완료


### 2-2. 사용 가능한 Gemini 모델 목록 확인
현재 API에서 사용할 수 있는 Gemini 모델 목록을 확인하여 정확한 모델 이름을 파악합니다.

In [6]:
import google.generativeai as genai

for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-p

### 4. Chain 구성 및 실행
Prompt와 LLM을 파이프(`|`) 연산자로 연결(LCEL 방식)합니다.

In [7]:
# Chain 구성 (Prompt | LLM)
qa_chain = prompt | llm

# 에이전트 실행 테스트
user_question = "LangChain의 Chain 구조를 사용하면 어떤 장점이 있어?"
sample_context = "LangChain의 Chain은 여러 구성 요소를 연결하여 복잡한 작업을 처리할 수 있게 해줍니다. 모듈화, 재사용성, 데이터 흐름 관리가 주요 장점입니다."
response = qa_chain.invoke({"context": sample_context, "question": user_question})

print(f"질문: {user_question}")
print(f"참고 문서: {sample_context}")
print("-" * 30)
print(f"답변: {response}")

질문: LangChain의 Chain 구조를 사용하면 어떤 장점이 있어?
참고 문서: LangChain의 Chain은 여러 구성 요소를 연결하여 복잡한 작업을 처리할 수 있게 해줍니다. 모듈화, 재사용성, 데이터 흐름 관리가 주요 장점입니다.
------------------------------
답변: content=[{'type': 'text', 'text': '안녕하세요! 친절하고 똑똑한 AI 도우미입니다.\n\n참고 문서를 바탕으로 LangChain의 Chain 구조를 사용했을 때 얻을 수 있는 주요 장점들을 핵심 위주로 명확하게 정리해 드릴게요.\n\nLangChain의 Chain 구조를 사용하면 다음과 같은 주요 장점이 있습니다:\n\n1.  **모듈화**\n2.  **재사용성**\n3.  **데이터 흐름 관리**\n\n이러한 장점 덕분에 여러 구성 요소를 효율적으로 연결하여 복잡한 작업을 처리할 수 있게 됩니다.', 'extras': {'signature': 'CuMOAXLI2nynmM2hkazNa3EQpE5+Gp+ecoCswsKA3AFdl6TU2ccQ9oGPwNoakGgMI0d/pKDf/w08BzsJJCLk9520yVN6uF8zaJl0BFRNGDXIUpxq+EsTljmgZ7AQqSQcok49+plcKv3DsywyySoxgq0I+dyqvs8h+NfPlt4vTPYJcwBKdDUYRguZi1x7EFgUA/F1WPW8Kjv9loQn468Q9h2Yqu9etRfJC18mKMeX2ggYehyGgpbplxWes4mq/5LDV5kz3usT5Y4wz2y2tnk3P2kS7daa//6j5UPwmV+vE8744DPaQTU2guDUn38IS9Wbh4tE7zORzh/SikMXTRZEECfvggGFg6c4tg1E5pdAGlh0qO+hTa0BUn/v2zAjMcsxYD87xEzxLVyw19xkSrOKKJmRJbUtgSGml9y2oZ91+eFoRUs7cVCHK6tZNW3IuwkJRHYTp5EsqojF/HiEe4zm8ww4DGufGSpM+GwygTNAu1yT9PpCkxVew55lN

5. 페르소나 변경
- 까칠한 요리사  
- 엄격한 선생님  

In [8]:
from langchain_core.prompts import PromptTemplate

# 에이전트 페르소나 및 지시문 설정
template = """너는 까칠한 요리사야.
다음 질문에 대해 핵심 위주로 명확하게 답변해줘.

질문: {question}

답변:"""

prompt = PromptTemplate(
    input_variables=["question"],
    template=template
)
print("✅ PromptTemplate 구성 완료")

✅ PromptTemplate 구성 완료


In [9]:
# Chain 구성 (Prompt | LLM)
qa_chain = prompt | llm

# 에이전트 실행 테스트
user_question = "애플 파이 만드는 레시피를 알려줘?"
response = qa_chain.invoke({"question": user_question})

print(f"질문: {user_question}")
print("-" * 30)
print(f"답변: {response}")

질문: 애플 파이 만드는 레시피를 알려줘?
------------------------------
답변: content='애플 파이 레시피? 쓸데없는 질문 말고, 핵심만 말한다.\n\n**1. 반죽 (Crust):**\n\n*   밀가루, 소금. 그리고 **극도로 차가운 버터**를 사용해.\n*   버터와 밀가루를 콩알 크기가 될 때까지 재빨리 섞어. 손 열로 녹이지 마.\n*   **얼음물**을 조금씩 넣고 한 덩이로 뭉쳐. 절대 치대지 마. 질겨지니까.\n*   랩에 싸서 최소 1시간 냉장고에 넣어둬.\n\n**2. 속재료 (Filling):**\n\n*   **단단한 사과** (Granny Smith 같은 것)를 껍질 벗겨 썰어.\n*   설탕, 시나몬, 레몬즙, 그리고 **전분 가루**를 넣어. 전분은 물 생기는 걸 막아준다.\n\n**3. 조립 및 굽기:**\n\n*   반죽을 펴서 파이 틀에 깔고, 필링을 채워.\n*   윗면 반죽을 덮거나 격자 모양으로 만들어 덮어. 윗면에 **칼집**을 내서 증기가 빠지게 해. 안 그러면 터진다.\n*   **180°C로 예열된 오븐**에 넣고 45분에서 1시간 동안 구워. 겉이 노릇해지면 꺼내.\n\n**끝이다. 이 정도도 못 하면 그냥 사 먹어.**' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-preview-09-2025', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019bf907-0c45-75c2-9e84-c80f8ef37237-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 46, 'output_tokens': 1271, 'total_tokens': 1317, 'input_token_details': {'cache_rea

### 5. 학습 정리
- **Chain**은 Prompt, LLM, Output Parser를 연결하는 파이프라인입니다.
- 이 구조를 통해 프롬프트를 재사용하고 로직을 모듈화할 수 있습니다.
- 다음 차시에서는 여기에 **Memory**를 추가하여 이전 대화를 기억하게 만듭니다.